# Advanced FileSet Features

This notebook demonstrates advanced FileSet capabilities:
- **Metadata filtering** — restrict which documents are used as seeds
- **RAG context generation** — retrieve supporting context with temporal constraints
- **RAG labeling** — resolve questions by searching the FileSet
- **Full combined pipeline** — context + labeling in one run

**Prerequisite**: Run `01_create_fileset.ipynb` first to create a FileSet and upload documents.

In [ ]:
%pip install lightningrod-ai python-dotenv pandas -q

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure FileSet ID

Paste the FileSet ID from notebook 1 below.

In [ ]:
fileset_id = "PASTE_YOUR_FILESET_ID_HERE"

In [4]:
import pandas as pd
from lightningrod import (
    QuestionPipeline,
    FileSetSeedGenerator,
    QdrantContextGenerator,
    QdrantRAGLabeler,
    QuestionGenerator,
    BinaryAnswerType,
    TemporalConstraint,
)

answer_type = BinaryAnswerType()

## Metadata Filtering

Use `metadata_filters` on the seed generator to restrict which files become seeds. Here we generate questions only from **APEX** documents.

In [8]:
pipeline_filtered = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
        metadata_filters=["ticker='APEX'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions="Generate yes/no questions about the specific financial metrics and business events in these quarterly reports.",
        questions_per_seed=2,
    ),
)

dataset_filtered = lr.transforms.run(
    pipeline_filtered,
    name="FileSet - APEX Only (Metadata Filter)",
)
print(f"Dataset: {dataset_filtered.id}")
print(f"Rows: {dataset_filtered.num_rows}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Job ID:           c7a090a9-96b5-4ab2-82e0-ed1f0b30e82b                                                       │
│                                                                                                                 │
│    Total cost: $0.00                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step                              ┃ Progress                ┃  In ┃  Out ┃  Rejected ┃  Errors ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ FileSetSeedGeneratorTransform     │ Complete                │   5 │    5 │         0 │       0 │       1s │  │
│  │ QuestionGeneratorTransform        │ Complete                │   5 │   10 │         0 │       0 │       3s │  │
│  └───────────────────────────────────┴─────────────────────────┴─────┴──────┴───────────┴─────────┴──────────┘  │
│                                                                                                                 │
│    View full details:                                                                                           │
│  ]8;id=351115;https://dashboard.lightningrod.ai/?redirect=/datasets/6daa38d5-0eea-4f48-b511-359386840cb0\https://dashboard.lightningrod.ai/?redirect=/datasets/6daa38d5-0eea-4f48-b511-359386840cb0]8;;\                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Dataset: 6daa38d5-0eea-4f48-b511-359386840cb0
Rows: 10


In [9]:
samples_filtered = dataset_filtered.download()
for i, s in enumerate(samples_filtered[:3]):
    print(f"--- Sample {i+1} ---")
    print(f"Seed (first 120 chars): {s.seed.seed_text[:120]}...")
    print(f"Question: {s.question.question_text}")
    print()

--- Sample 1 ---
Seed (first 120 chars): APEX Technologies Inc. — Quarterly Investor Report, Q2 2024
Period ending June 30, 2024

Financial Highlights:
- Revenue...
Question: Will APEX Technologies Inc. officially announce the completion of its Northern Virginia data center expansion on or before December 31, 2024?

--- Sample 2 ---
Seed (first 120 chars): APEX Technologies Inc. — Quarterly Investor Report, Q2 2024
Period ending June 30, 2024

Financial Highlights:
- Revenue...
Question: Will APEX Technologies Inc. report total revenue of at least $2.35 billion for the third quarter of 2024?

--- Sample 3 ---
Seed (first 120 chars): APEX Technologies Inc. — Quarterly Investor Report, Q4 2024
Period ending December 31, 2024

Financial Highlights:
- Rev...
Question: Will APEX Technologies Inc. complete the acquisition of CyberShield Corp on or before March 31, 2025?



## RAG Context Generation

`FileSetContextGenerator` retrieves supporting context from the FileSet for each generated question.

- **`metadata_filter_keys=["ticker"]`** — only retrieve context from the same company
- **`temporal_constraint=BEFORE`** — only retrieve context from documents dated before the seed, preventing lookahead bias

## RAG Context Generation

`QdrantContextGenerator` retrieves supporting context from the FileSet for each generated question.

- **`metadata_filter_keys=["ticker"]`** — only retrieve context from the same company
- **`temporal_constraint=BEFORE`** — only retrieve context from documents dated before the seed, preventing lookahead bias

## RAG Labeling

`FileSetRAGLabeler` resolves questions by searching the FileSet for answers.

- **`temporal_constraint=AFTER`** — only search documents dated after the seed, so forward-looking questions are resolved by later reports
- **`confidence_threshold=0.7`** — only label questions where the labeler is at least 70% confident

## RAG Labeling

`QdrantRAGLabeler` resolves questions by searching the FileSet for answers.

- **`temporal_constraint=AFTER`** — only search documents dated after the seed, so forward-looking questions are resolved by later reports
- **`confidence_threshold=0.7`** — only label questions where the labeler is at least 70% confident

In [ ]:
pipeline_labeler = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
        metadata_filters=["ticker='VGI'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions=(
            "Generate yes/no questions about forward-looking statements, guidance, and planned initiatives "
            "mentioned in these quarterly reports. Focus on questions whose answers would be found in "
            "subsequent quarterly reports."
        ),
        questions_per_seed=2,
    ),
    labeler=QdrantRAGLabeler(
        file_set_id=fileset_id,
        metadata_filter_keys=["ticker"],
        temporal_constraint=TemporalConstraint.AFTER,
        confidence_threshold=0.7,
        answer_type=answer_type,
    ),
)

dataset_labeler = lr.transforms.run(
    pipeline_labeler,
    max_questions=6,
    name="FileSet - RAG Labeler (VGI, AFTER)",
)
print(f"Dataset: {dataset_labeler.id}")
print(f"Rows: {dataset_labeler.num_rows}")

## Full Pipeline — Context + Labeling

Combine context generation and labeling in a single pipeline:
- **Context** (`BEFORE`) — retrieve earlier reports as supporting context
- **Labeler** (`AFTER`) — resolve forward-looking questions using later reports

In [16]:
pipeline_full = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions=(
            "Generate yes/no questions about forward-looking statements and guidance in these investor reports. "
            "Focus on questions that can be verified by looking at later quarterly reports for the same company."
        ),
        questions_per_seed=1,
    ),
    context_generators=[
        FileSetContextGenerator(
            file_set_id=fileset_id,
            metadata_filter_keys=["ticker"],
            temporal_constraint=TemporalConstraint.BEFORE,
        ),
    ],
    labeler=FileSetRAGLabeler(
        file_set_id=fileset_id,
        metadata_filter_keys=["ticker"],
        temporal_constraint=TemporalConstraint.AFTER,
        confidence_threshold=0.7,
        answer_type=answer_type,
    ),
)

dataset_full = lr.transforms.run(
    pipeline_full,
    max_questions=8,
    name="FileSet - Full Pipeline (Context + Labeler)",
)
print(f"Dataset: {dataset_full.id}")
print(f"Rows: {dataset_full.num_rows}")

/usr/local/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Dataset: f847bcc6-a34e-416b-86e4-d3a11ecd9a76
Rows: 8


In [ ]:
pipeline_full = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions=(
            "Generate yes/no questions about forward-looking statements and guidance in these investor reports. "
            "Focus on questions that can be verified by looking at later quarterly reports for the same company."
        ),
        questions_per_seed=1,
    ),
    context_generators=[
        QdrantContextGenerator(
            file_set_id=fileset_id,
            metadata_filter_keys=["ticker"],
            temporal_constraint=TemporalConstraint.BEFORE,
        ),
    ],
    labeler=QdrantRAGLabeler(
        file_set_id=fileset_id,
        metadata_filter_keys=["ticker"],
        temporal_constraint=TemporalConstraint.AFTER,
        confidence_threshold=0.7,
        answer_type=answer_type,
    ),
)

dataset_full = lr.transforms.run(
    pipeline_full,
    max_questions=8,
    name="FileSet - Full Pipeline (Context + Labeler)",
)
print(f"Dataset: {dataset_full.id}")
print(f"Rows: {dataset_full.num_rows}")

In [18]:
rows = dataset_full.flattened()
df = pd.DataFrame(rows)

cols = ["question_text", "label", "label_confidence", "reasoning", "is_valid"]
df[[c for c in cols if c in df.columns]]

,question_text,label,label_confidence,reasoning
0,Will APEX Technologies Inc. close the acquisit...,1.0,1.0,"The RAG answer explicitly states, 'Yes, APEX T..."
1,Will APEX Technologies Inc. report Q2 2025 rev...,NaN,1.0,The RAG Answer explicitly states that there is...
2,Will APEX Technologies Inc. report Q3 2024 rev...,1.0,1.0,The RAG answer explicitly states that APEX Tec...
3,Did APEX Technologies Inc. report revenue of a...,1.0,1.0,"The RAG answer explicitly states 'Yes, APEX Te..."
4,Will APEX Technologies Inc. report a revenue o...,1.0,1.0,The RAG answer explicitly states that APEX Tec...
5,Will Vanguard Industries Inc. report a total r...,1.0,1.0,The RAG answer explicitly states that Vanguard...
6,Will Vanguard Industries Inc. report total rev...,NaN,1.0,The RAG answer explicitly states there is no i...
7,Will Vanguard Industries Inc. report total rev...,1.0,1.0,"The RAG answer explicitly states 'Yes, Vanguar..."
